<a href="https://colab.research.google.com/github/Ebrardemir/amazon-sentiment-analysis/blob/main/notebooks/01_Veri_Okuma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Amazon Dataset Reviews 2023 Verilerinden Duygu Analizi
Bu projede, Amazon Reviews 2023  Dateseti içerisinden aldığımız Electronic kategorisinin verileri için duygu analizi yapıyoruz.

In [ ]:
import gzip
import json
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
input_path = '/content/drive/MyDrive/Veri_madenciliği/Dataset/Electronics.jsonl.gz'

In [ ]:
metadata_path = '/content/drive/MyDrive/Veri_madenciliği/Dataset/meta_Electronics.jsonl.gz'

# Satır ve Sütun Analizi
Aşağıdaki hücrede kullandığımız Electronics.json.gz dosyasında kaç satır ,kaç sütun ve hangi sütunların bulunduğunun analizini yapıyoruz.

In [ ]:
def analysis_jsonl_gz(file_path):

    #Belirtilen .jsonl.gz dosyasını tarar, toplam satır sayısını ve sütun isimlerini döndürür.

    row_count = 0
    columns = []

    print(f"Dosya tarama işlemi başladı: {file_path}")
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        for line in f:
            if row_count == 0:
                try:
                    # Sütun isimlerini alalım
                    columns = list(json.loads(line).keys())
                except json.JSONDecodeError:
                    print(f"Hata: İlk satır JSON olarak ayrıştırılamadı. {file_path}")
                    return 0, []
            row_count += 1

    print(f"\n- ANALİZ SONUCU - ({file_path})")
    print(f"Toplam Satır Sayısı: {row_count:,}")
    print(f"Toplam Sütun Sayısı: {len(columns)}")
    print(f"Sütun İsimleri: {columns}")
    return row_count, columns

In [ ]:
# input_path için fonksiyonu kullanma
print("\n*** input_path (Electronics.jsonl.gz) Analizi ***")
input_row_count, input_columns = analysis_jsonl_gz(input_path)

'meta_Electronics.jsonl.gz' bu dosyada ürünle ilgili bilgiler bulunur.Hangi sütunların bulunduğu aşağıda listelenmiştir.

In [ ]:
# metadata_path için fonksiyonu kullanma
print("\n*** metadata_path (meta_Electronics.jsonl.gz) Analizi ***")
metadata_row_count, metadata_columns = analysis_jsonl_gz(metadata_path)

Verinin içeriğini görmek için ilk 5 satırı ekrana bastırdık.

In [ ]:
import pandas as pd
import gzip
import json


def preview_jsonl_gz_first_n_rows(file_path, num_rows=5):

    preview_data = []
    try:
        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
            for i, line in enumerate(f):
                preview_data.append(json.loads(line))
                if i == num_rows - 1:
                    break
        df_preview = pd.DataFrame(preview_data)
        pd.set_option('display.max_columns', None)
        print(f"*** Dosya: {file_path} İlk {num_rows} Satır ***")
        display(df_preview)
        return df_preview
    except Exception as e:
        print(f"Dosya okuma veya işleme hatası ({file_path}): {e}")
        return None

In [ ]:
# input_path dosyasının ilk 5 satırını görüntülemek için fonksiyonu kullanma
input_df_preview = preview_jsonl_gz_first_n_rows(input_path, num_rows=5)



In [ ]:
# metadata_path dosyasının ilk 5 satırını da görüntüleyelim
metadata_df_preview = preview_jsonl_gz_first_n_rows(metadata_path, num_rows=20)

# Rating Yorum Grafiği
Bu bölümde, elimizdeki ham veri setindeki ürün yorumlarının yıldız (rating) dağılımını görselleştiriyoruz. Bu grafik, kullanıcıların ürünlere verdiği puanların genel eğilimini ve hangi yıldız puanlarının daha yoğun olduğunu anlamamızı sağlayacaktır. Özellikle 1, 2, 3, 4 ve 5 yıldızlı yorumların sayısal dağılımını bar grafik üzerinde görebilirsiniz.

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gzip
import json

def plot_rating_distribution(
    file_path,
    title='Yıldız Puanı Dağılımı',
    output_dir=None,
    output_filename='rating_dagilimi_analizi.png'
):

    rating_counts = Counter()
    print(f"Dosya tarıyor ve rating dağılımı sayılıyor: {file_path}")

    try:
        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
            for i, line in enumerate(f):
                data = json.loads(line)
                r = data.get('rating')
                if r is not None:
                    rating_counts[r] += 1

    except Exception as e:
        print(f"Hata oluştu: {e}")
        return {}

    if not rating_counts:
        print(f"Uyarı: '{file_path}' dosyasında 'rating' verisi bulunamadı.")
        return {}

    ratings = sorted(rating_counts.keys())
    counts = [rating_counts[r] for r in ratings]

    plt.figure(figsize=(10, 6))
    sns.barplot(x=ratings, y=counts, palette='viridis', edgecolor='black')

    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Rating', fontsize=12)
    plt.ylabel('Yorum Sayısı', fontsize=12)

    # Y ekseni formatını milyon olarak ayarla ve "Milyon" kelimesini ekle
    plt.gca().get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, p: f"{int(x/1e6):,} Milyon"))

    plt.grid(axis='y', linestyle='--', alpha=0.7)

    if output_dir:
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        full_output_path = os.path.join(output_dir, output_filename)
        plt.savefig(full_output_path, dpi=300)
        print(f"Grafik kaydedildi: {full_output_path}")

    plt.show()

    print("\n*** ANALİZ SONUÇLARI (Rating Dağılımı) ***")
    for r, count in sorted(rating_counts.items()):
        print(f"{r} Yıldız: {count:,} yorum")

    return rating_counts

# input_path için fonksiyonu kullanma
plot_output_dir = '/content/drive/MyDrive/Veri_madenciliği/Grafik/'
input_rating_counts = plot_rating_distribution(
    input_path,
    title='Ham Veri Seti Yıldız Puanı Dağılımı',
    output_dir=plot_output_dir,
    output_filename='rating_dagilimi_analizi_input.png'
)

In [ ]:
def display_and_save_rating_table(
    rating_counts,
    output_dir,
    output_filename='rating_dagilimi_tablosu.csv'
):

    # input_rating_counts'ı DataFrame'e dönüştür
    df_rating_counts = pd.DataFrame(rating_counts.items(), columns=['Rating', 'Yorum Sayısı'])
    df_rating_counts = df_rating_counts.sort_values(by='Rating').reset_index(drop=True)

    print("\n*** Yıldız Puanı Dağılımı Tablosu ***")
    display(df_rating_counts)

    # Tabloyu CSV olarak kaydet
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    full_output_path = os.path.join(output_dir, output_filename)
    df_rating_counts.to_csv(full_output_path, index=False, encoding='utf-8-sig')
    print(f"Tablo kaydedildi: {full_output_path}")
    return df_rating_counts

In [ ]:
table_output_dir = '/content/drive/MyDrive/Veri_madenciliği/Grafik/'
input_df_rating_counts = display_and_save_rating_table(
    input_rating_counts,
    output_dir=table_output_dir,
    output_filename='rating_dagilimi_tablosu_input.csv'
)

## Ana Kategori Rating Analiz Fonksiyonu

Bu bölümde, ana kategoriye göre yıldız puanı dağılımını ve toplam yorum sayısını analiz eden bir fonksiyon oluşturulmuştur. Bu fonksiyon, verilen inceleme ve metadata dosyalarını işleyerek, her bir ana kategori için 0'dan 5'e kadar olan her bir yıldız puanının sayısını ve toplam yorum sayısını içeren bir DataFrame döndürür ve ayrıca bu DataFrame'i belirtilen bir yola CSV olarak kaydeder.

In [ ]:
from collections import defaultdict
import pandas as pd
import gzip
import json
import os

def analysis_main_category_ratings(
    input_file_path,
    metadata_file_path,
    output_dir,
    output_filename='ana_kategori_yorum_ve_rating_ozeti.csv'
):
    print("Ana kategori rating analizi başladı.")

    # Step 1: Create a mapping from parent_asin to main_category from metadata file
    asin_to_category_map = {}
    try:
        with gzip.open(metadata_file_path, 'rt', encoding='utf-8') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    parent_asin = data.get('parent_asin')
                    main_category = data.get('main_category')

                    if parent_asin and main_category and main_category != 'None':
                        asin_to_category_map[parent_asin] = main_category
                except json.JSONDecodeError:
                    pass
    except Exception as e:
        # print(f"Metadata okuma hatası: {e}") # Keep this for error reporting
        return pd.DataFrame()

    # Step 2: Process input_path to count ratings per main_category
    category_rating_counts = defaultdict(lambda: defaultdict(int))
    try:
        with gzip.open(input_file_path, 'rt', encoding='utf-8') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    parent_asin = data.get('parent_asin')
                    rating = data.get('rating')

                    if parent_asin and rating is not None:
                        main_category = asin_to_category_map.get(parent_asin)
                        if main_category:
                            category_rating_counts[main_category][float(rating)] += 1
                except json.JSONDecodeError:
                    pass

    except Exception as e:

        return pd.DataFrame()


    data_for_df = []
    for category, ratings in category_rating_counts.items():
        row = {'Main Category': category}
        total_reviews = 0
        for r in range(6):  # Rating'ler 0.0'dan 5.0'a kadar
            rating_key = float(r)
            count = ratings.get(rating_key, 0)
            row[f'Rating_{r}.0'] = count
            total_reviews += count
        row['Total Reviews'] = total_reviews
        data_for_df.append(row)

    df_main_category_summary = pd.DataFrame(data_for_df)

    if not df_main_category_summary.empty:
        # Toplam Yorum Sayısı'na göre sırala
        df_main_category_summary = df_main_category_summary.sort_values(by='Total Reviews', ascending=False).reset_index(drop=True)


    display(df_main_category_summary)


    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    full_output_path = os.path.join(output_dir, output_filename)
    df_main_category_summary.to_csv(full_output_path, index=False, encoding='utf-8-sig')

    print("Ana kategori rating analizi tamamlandı.")
    return df_main_category_summary

### Fonksiyonu Kullanarak Analiz Gerçekleştirme
Şimdi oluşturduğumuz `analysis_main_category_ratings` fonksiyonunu mevcut `input_path` ve `metadata_path` değişkenlerimizle çağırarak analizi gerçekleştirelim.

In [ ]:
# Analiz çıktı dizini
analysis_output_dir = '/content/drive/MyDrive/Veri_madenciliği/Dataset/'

# Fonksiyonu çağır
df_main_category_summary = analysis_main_category_ratings(
    input_file_path=input_path,
    metadata_file_path=metadata_path,
    output_dir=analysis_output_dir,
    output_filename='ana_kategori_yorum_ve_rating_ozeti_yeniden.csv'
)

# Kategori Filtreleme
Bu bölümde, örneklem dosyamız için her bir ana kategorinin belirli rating değerleri (1, 2,3, 4 ve 5) için minimum 10.000 yoruma sahip olmasını sağlayarak filtreleme yapıyoruz. Bu, duygu analizi için yeterli ve dengeli veri içeren kategorileri belirlememizi sağlar. 10.000 yorumun altında kalan rating değerlerine sahip kategoriler elenecektir.

In [ ]:
import pandas as pd
import os

def filter_main_categories_by_min_rating_count(
    df_main_category_summary,
    min_count_per_rating=2000,
    output_dir=None,
    output_filename='elenmis_main_kategoriler.csv'
):


    # Girdi kontrolü
    if df_main_category_summary is None or df_main_category_summary.empty:
        print("HATA: df_main_category_summary boş veya None.")
        return pd.DataFrame(), pd.DataFrame()

    if 'Main Category' not in df_main_category_summary.columns:
        print("HATA: Tabloda 'Main Category' sütunu yok.")
        return pd.DataFrame(), pd.DataFrame()

    # Rating sütun adlarını kontrol et (Rating_1.0, Rating_2.0 formatı)
    rating_columns = ['Rating_1.0', 'Rating_2.0', 'Rating_4.0', 'Rating_5.0']
    missing_cols = [col for col in rating_columns if col not in df_main_category_summary.columns]
    if missing_cols:
        print(f"HATA: Şu sütunlar tabloda yok: {missing_cols}")
        print(f"Mevcut sütunlar: {df_main_category_summary.columns.tolist()}")
        return pd.DataFrame(), pd.DataFrame()

    print(f"Main kategori eleme başladı.")
    print(f"Eşik: Rating 1, 2, 4, 5'in her birinden en az {min_count_per_rating:,} yorum")
    print(f"(Rating_3 duygu analizinde kullanılmayacağı için kontrole dahil değil)")
    print(f"Başlangıç main kategori sayısı: {len(df_main_category_summary)}")

    # Her rating sütununun eşiği geçip geçmediğini kontrol et
    # .all(axis=1) -> tek bir rating bile eşiğin altındaysa kategori elenecek
    mask = (df_main_category_summary[rating_columns] >= min_count_per_rating).all(axis=1)

    df_passed = df_main_category_summary[mask].copy().reset_index(drop=True)
    df_failed = df_main_category_summary[~mask].copy().reset_index(drop=True)

    # Özet bilgi
    print(f"\n*** ELEME SONUCU ***")
    print(f"Eşiği geçen main kategori sayısı: {len(df_passed)}")
    print(f"Elenen main kategori sayısı: {len(df_failed)}")

    if not df_passed.empty:
        # Toplam yorum (tüm rating'ler)
        total_reviews_passed = df_passed['Total Reviews'].sum() if 'Total Reviews' in df_passed.columns else 0
        print(f"Geçen kategorilerdeki toplam yorum (tüm rating'ler): {total_reviews_passed:,}")

        # Rating_3 hariç kullanılabilir toplam (örneklem havuzuna girecek)
        usable_total = df_passed[rating_columns].sum().sum()
        print(f"Geçen kategorilerde kullanılabilir yorum (Rating_3 hariç): {usable_total:,}")

    print(f"\n--- EŞİĞİ GEÇEN MAIN KATEGORİLER ---")
    display(df_passed)

    # Elenen main kategorilerde hangi rating'in eksik olduğunu göster
    if not df_failed.empty:
        print(f"\n*** ELENEN MAIN KATEGORİLER (neden elendiği ile birlikte) ***")
        df_failed_display = df_failed.copy()

        def find_missing_ratings(row):
            missing = []
            for col in rating_columns:
                if row[col] < min_count_per_rating:
                    missing.append(f"{col}={int(row[col]):,}")
            return ', '.join(missing)

        df_failed_display['Eksik Ratings'] = df_failed_display.apply(find_missing_ratings, axis=1)
        display(df_failed_display)

    # CSV kaydet
    if output_dir:
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        passed_path = os.path.join(output_dir, 'gecen_' + output_filename)
        failed_path = os.path.join(output_dir, 'elenen_' + output_filename)
        df_passed.to_csv(passed_path, index=False, encoding='utf-8-sig')
        df_failed.to_csv(failed_path, index=False, encoding='utf-8-sig')
        print(f"\nGeçen main kategoriler: {passed_path}")
        print(f"Elenen main kategoriler: {failed_path}")

    print("\nEleme tamamlandı.")
    return df_passed, df_failed

In [ ]:
df_main_passed, df_main_failed = filter_main_categories_by_min_rating_count(
    df_main_category_summary=df_main_category_summary,
    min_count_per_rating=10000,
    output_dir='/content/drive/MyDrive/Veri_madenciliği/Dataset/',
    output_filename='main_kategori_eleme.csv'
)

# Kategori Yorumlama
Filtrelemeden geçen kategorilerin yorum ve ürün sayılarını bularak sistemde kullanacağımız limit değerlerini belirlemek için bir kod yapısı kurduk.

In [ ]:
from collections import defaultdict
import pandas as pd
import gzip
import json
import os

def analyze_products_per_main_category(
    df_main_passed,
    input_file_path,
    metadata_file_path,
    output_dir=None,
    output_filename='main_kategori_urun_istatistikleri.csv'
):


    # Girdi kontrolü
    if df_main_passed is None or df_main_passed.empty:
        print("HATA: df_main_passed boş veya None. Önce main kategori eleme yapmalısın.")
        return pd.DataFrame()

    if 'Main Category' not in df_main_passed.columns:
        print("HATA: df_main_passed içinde 'Main Category' sütunu yok.")
        return pd.DataFrame()

    # Sadece eşiği geçen main kategorilerle çalışacağız
    valid_main_categories = set(df_main_passed['Main Category'].tolist())
    print(f"Analiz edilecek main kategori sayısı: {len(valid_main_categories)}")

    # ---- 1. ADIM: Metadata'dan parent_asin -> main_category eşlemesi ----
    print("\n[1/2] Metadata okunuyor...")
    asin_to_main_category = {}

    try:
        with gzip.open(metadata_file_path, 'rt', encoding='utf-8') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    parent_asin = data.get('parent_asin')
                    main_category = data.get('main_category')

                    if not parent_asin:
                        continue

                    if not main_category or not str(main_category).strip() or main_category == 'None':
                        continue

                    main_category = str(main_category).strip()

                    if main_category in valid_main_categories:
                        asin_to_main_category[parent_asin] = main_category

                except json.JSONDecodeError:
                    pass
    except Exception as e:
        print(f"Metadata okuma hatası: {e}")
        return pd.DataFrame()

    print(f"Eşiği geçen main kategorilere ait toplam ürün sayısı: {len(asin_to_main_category):,}")

    # ---- 2. ADIM: Yorumları oku, her ürünün yorum sayısını hesapla ----
    print("\n[2/2] Yorumlar okunuyor, ürün başı yorum sayıları hesaplanıyor...")

    product_review_counts = defaultdict(lambda: defaultdict(int))

    try:
        with gzip.open(input_file_path, 'rt', encoding='utf-8') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    parent_asin = data.get('parent_asin')

                    if parent_asin:
                        main_cat = asin_to_main_category.get(parent_asin)
                        if main_cat:
                            product_review_counts[main_cat][parent_asin] += 1

                except json.JSONDecodeError:
                    pass
    except Exception as e:
        print(f"Yorum dosyası okuma hatası: {e}")
        return pd.DataFrame()

    # ---- 3. ADIM: İstatistikleri hesapla ----
    print("\nİstatistikler hesaplanıyor...")

    stats_data = []
    for main_cat in valid_main_categories:
        products = product_review_counts.get(main_cat, {})

        if not products:
            stats_data.append({
                'Main Category': main_cat,
                'Ürün Sayısı': 0,
                'Toplam Yorum': 0,
                'Ortalama Yorum/Ürün': 0,
                'Medyan Yorum/Ürün': 0,
                'Min Yorum/Ürün': 0,
                'Max Yorum/Ürün': 0,
                'En Az 5 Yorumu Olan Ürün': 0,
                'En Az 10 Yorumu Olan Ürün': 0,
                'En Az 50 Yorumu Olan Ürün': 0,
                'En Az 100 Yorumu Olan Ürün': 0,
                'En Az 1000 Yorumu Olan Ürün': 0,
            })
            continue

        review_counts = list(products.values())
        review_counts_series = pd.Series(review_counts)

        stats_data.append({
            'Main Category': main_cat,
            'Ürün Sayısı': len(products),
            'Toplam Yorum': sum(review_counts),
            'Ortalama Yorum/Ürün': round(review_counts_series.mean(), 1),
            'Medyan Yorum/Ürün': int(review_counts_series.median()),
            'Min Yorum/Ürün': int(review_counts_series.min()),
            'Max Yorum/Ürün': int(review_counts_series.max()),
            'En Az 5 Yorumu Olan Ürün': int((review_counts_series >= 5).sum()),
            'En Az 10 Yorumu Olan Ürün': int((review_counts_series >= 10).sum()),
            'En Az 50 Yorumu Olan Ürün': int((review_counts_series >= 50).sum()),
            'En Az 100 Yorumu Olan Ürün': int((review_counts_series >= 100).sum()),
            'En Az 1000 Yorumu Olan Ürün': int((review_counts_series >= 1000).sum()),
        })

    df_stats = pd.DataFrame(stats_data)
    df_stats = df_stats.sort_values(by='Ürün Sayısı', ascending=False).reset_index(drop=True)


    print(f"\n*** ANALİZ SONUÇLARI ***")
    print(f"Analiz edilen main kategori sayısı: {len(df_stats)}")
    print(f"Toplam ürün sayısı: {df_stats['Ürün Sayısı'].sum():,}")
    print(f"Toplam yorum sayısı: {df_stats['Toplam Yorum'].sum():,}")
    print(f"Ortalama main kategori başı ürün sayısı: {df_stats['Ürün Sayısı'].mean():.0f}")
    print(f"Medyan main kategori başı ürün sayısı: {df_stats['Ürün Sayısı'].median():.0f}")

    print(f"\n*** TÜM MAIN KATEGORİLER (ürün sayısına göre sıralı) ***")
    display(df_stats)

    # CSV kaydet
    if output_dir:
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        full_output_path = os.path.join(output_dir, output_filename)
        df_stats.to_csv(full_output_path, index=False, encoding='utf-8-sig')
        print(f"\nKaydedildi: {full_output_path}")

    print("\nAnaliz tamamlandı.")
    return df_stats

In [ ]:
df_main_product_stats = analyze_products_per_main_category(
    df_main_passed=df_main_passed,
    input_file_path=input_path,
    metadata_file_path=metadata_path,
    output_dir='/content/drive/MyDrive/Veri_madenciliği/Dataset/',
    output_filename='main_kategori_urun_istatistikleri.csv'
)

# Örneklem Oluşturma

Bu adımda, duygu analizi için dengeli bir örneklem oluşturuyoruz. Bu süreçte aşağıdaki filtreler ve limitler uygulanmıştır:

*   **Ürün Başı Minimum Yorum Sayısı:** Sadece en az 5 yoruma sahip ürünler örnekleme dahil edilmiştir (`min_reviews_per_product=5`). Bu, yeterli içeriğe sahip ürünleri hedeflememizi sağlar.
*   **Ürün Başı Maksimum Yorum Sayısı:** Her bir üründen maksimum 50 yorum alınmıştır (`max_reviews_per_product=50`). Bu, belirli bir ürünün örneklemi domine etmesini engeller ve çeşitliliği artırır.
*   **Kategori Havuzları:** Belirlenen kriterlere uyan yorumlar, her ana kategori için ayrı havuzlarda toplanmıştır.
*   **Rating Başı Hedef Örnek Sayısı:** Bu havuzlardan, her ana kategori ve her rating değeri (1, 2, 4 ve 5 yıldız) için 10.000'er yorum rastgele seçilmesi hedeflenmiştir (`target_samples_per_rating=10000`). Bu, her duygu kutusu için dengeli bir dağılım sağlar.
*   **Hariç Tutulan Kategoriler:** 'Apple Products' kategorisindeki yorumlar, çeşitlilik sağlamadığı ve örneklem sınırlarında kaldığı için analizden hariç tutulmuştur.

In [ ]:
from collections import defaultdict
import pandas as pd
import gzip
import json
import os
import random

def create_main_category_sample(
    df_main_passed,
    input_file_path,
    metadata_file_path,
    output_dir,
    output_filename='main_kategori_orneklem.csv',
    target_samples_per_rating=10000,
    min_reviews_per_product=5,
    max_reviews_per_product=50,
    excluded_ratings=[3],
    excluded_categories=['Apple Products'],
    random_seed=42
):

    # Reproducibility
    random.seed(random_seed)

    # ===== GİRDİ KONTROLÜ =====
    if df_main_passed is None or df_main_passed.empty:
        print("HATA: df_main_passed boş veya None.")
        return pd.DataFrame()

    if 'Main Category' not in df_main_passed.columns:
        print("HATA: df_main_passed içinde 'Main Category' sütunu yok.")
        return pd.DataFrame()

    # Hedef kategorileri belirle
    all_categories = set(df_main_passed['Main Category'].tolist())
    target_categories = all_categories - set(excluded_categories)

    print("=" * 70)
    print("ÖRNEKLEM ÇEKME BAŞLADI")
    print("=" * 70)
    print(f"Hedef kategori sayısı: {len(target_categories)}")
    print(f"Hariç tutulan kategoriler: {excluded_categories}")
    print(f"Her rating için hedef: {target_samples_per_rating:,}")
    print(f"Kategori başı hedef toplam: {target_samples_per_rating * 4:,}")
    print(f"Atlanan rating'ler: {excluded_ratings}")
    print(f"Min yorum/ürün: {min_reviews_per_product}")
    print(f"Max yorum/ürün: {max_reviews_per_product}")
    print(f"Random seed: {random_seed}")
    print()

    #  ADIM 1: METADATA OKUMA
    print("[1/4] Metadata okunuyor, parent_asin -> main_category eşlemesi çıkarılıyor...")
    asin_to_main_category = {}

    try:
        with gzip.open(metadata_file_path, 'rt', encoding='utf-8') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    parent_asin = data.get('parent_asin')
                    main_category = data.get('main_category')

                    if not parent_asin:
                        continue

                    if not main_category or not str(main_category).strip() or main_category == 'None':
                        continue

                    main_category = str(main_category).strip()

                    # Sadece hedef kategorilere ait ürünleri tut
                    if main_category in target_categories:
                        asin_to_main_category[parent_asin] = main_category

                except json.JSONDecodeError:
                    pass
    except Exception as e:
        print(f"Metadata okuma hatası: {e}")
        return pd.DataFrame()

    print(f"  → Hedef kategorilere ait toplam ürün: {len(asin_to_main_category):,}")

    #  ADIM 2: ÜRÜN YORUM SAYILARI
    print(f"\n[2/4] 1. geçiş: Ürün başına yorum sayıları hesaplanıyor...")
    product_total_counts = defaultdict(int)

    try:
        with gzip.open(input_file_path, 'rt', encoding='utf-8') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    parent_asin = data.get('parent_asin')

                    if parent_asin and parent_asin in asin_to_main_category:
                        product_total_counts[parent_asin] += 1

                except json.JSONDecodeError:
                    pass
    except Exception as e:
        print(f"1. geçiş okuma hatası: {e}")
        return pd.DataFrame()

    # Min eşiği geçen ürünleri belirle
    valid_products = {
        asin for asin, count in product_total_counts.items()
        if count >= min_reviews_per_product
    }

    print(f"  → Toplam ürün: {len(product_total_counts):,}")
    print(f"  → Min {min_reviews_per_product} yorum eşiğini geçen ürün: {len(valid_products):,}")
    print(f"  → Elenen ürün: {len(product_total_counts) - len(valid_products):,}")

    #  ADIM 3: İKİNCİ GEÇİŞ - HAVUZ OLUŞTURMA
    print(f"\n[3/4] 2. geçiş: Havuz oluşturuluyor (ürün başı max {max_reviews_per_product} yorum)...")


    pools = defaultdict(lambda: defaultdict(list))

    # Ürün başı alınan yorum sayacı
    product_taken_counts = defaultdict(int)

    excluded_ratings_set = set(excluded_ratings)
    processed_count = 0
    added_to_pool_count = 0

    try:
        with gzip.open(input_file_path, 'rt', encoding='utf-8') as f:
            for line in f:
                try:
                    data = json.loads(line)
                    parent_asin = data.get('parent_asin')

                    # Filtre 1: Hedef kategoride mi?
                    if not parent_asin or parent_asin not in asin_to_main_category:
                        continue

                    # Filtre 2: Min yorum eşiğini geçiyor mu?
                    if parent_asin not in valid_products:
                        continue

                    # Filtre 3: Rating kontrolü
                    rating = data.get('rating')
                    if rating is None:
                        continue

                    try:
                        rating_int = int(float(rating))
                    except (ValueError, TypeError):
                        continue

                    # Atlanacak rating mi?
                    if rating_int in excluded_ratings_set:
                        continue

                    # 1-5 arasında mı?
                    if rating_int < 1 or rating_int > 5:
                        continue

                    # Filtre 4: Ürün başı limit dolu mu?
                    if product_taken_counts[parent_asin] >= max_reviews_per_product:
                        continue

                    # Havuza ekle
                    main_cat = asin_to_main_category[parent_asin]

                    review_record = {
                        'main_category': main_cat,
                        'parent_asin': parent_asin,
                        'user_id': data.get('user_id', ''),
                        'rating': rating_int,
                        'title': data.get('title', ''),
                        'text': data.get('text', ''),
                        'timestamp': data.get('timestamp', 0),
                        'helpful_vote': data.get('helpful_vote', 0),
                        'verified_purchase': data.get('verified_purchase', False),
                    }

                    pools[main_cat][rating_int].append(review_record)
                    product_taken_counts[parent_asin] += 1
                    added_to_pool_count += 1

                    processed_count += 1

                except json.JSONDecodeError:
                    pass
                except Exception as inner_e:
                    pass

    except Exception as e:
        print(f"2. geçiş okuma hatası: {e}")
        return pd.DataFrame()

    print(f"  → Havuza eklenen toplam yorum: {added_to_pool_count:,}")

    # Havuz özetini yazdır
    print(f"\n  Havuz dağılımı:")
    print(f"  {'Kategori':<35} {'R1':>10} {'R2':>10} {'R4':>10} {'R5':>10}")
    print(f"  {'-'*35} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")
    for cat in sorted(target_categories):
        r1 = len(pools[cat].get(1, []))
        r2 = len(pools[cat].get(2, []))
        r4 = len(pools[cat].get(4, []))
        r5 = len(pools[cat].get(5, []))
        print(f"  {cat[:35]:<35} {r1:>10,} {r2:>10,} {r4:>10,} {r5:>10,}")

    # ===== ADIM 4: RASTGELE ÖRNEKLEM SEÇİMİ =====
    print(f"\n[4/4] Her kategori × rating için rastgele {target_samples_per_rating:,} yorum seçiliyor...")

    all_samples = []
    warnings = []

    for cat in sorted(target_categories):
        for rating in [1, 2, 4, 5]:
            pool = pools[cat].get(rating, [])
            pool_size = len(pool)

            if pool_size == 0:
                warnings.append(f"  UYARI: {cat} - Rating_{rating} havuzu boş!")
                continue

            if pool_size < target_samples_per_rating:
                warnings.append(
                    f"  UYARI: {cat} - Rating_{rating} havuzunda sadece {pool_size:,} yorum var "
                    f"(hedef: {target_samples_per_rating:,}). Tümü alındı."
                )
                selected = pool
            else:
                selected = random.sample(pool, target_samples_per_rating)

            all_samples.extend(selected)

    # Uyarıları yazdır
    if warnings:
        print("\n  Uyarılar:")
        for w in warnings:
            print(w)
    else:
        print(f"  → Tüm kategori × rating kombinasyonlarında hedef sayıya ulaşıldı.")

    # ===== DATAFRAME OLUŞTUR =====
    print(f"\nDataFrame oluşturuluyor...")
    df_sample = pd.DataFrame(all_samples)

    # Sütun sırasını düzenle
    column_order = [
        'main_category', 'parent_asin', 'user_id', 'rating',
        'title', 'text', 'timestamp', 'helpful_vote', 'verified_purchase'
    ]
    df_sample = df_sample[column_order]


    print(f"\n" + "=" * 70)
    print("ÖRNEKLEM SONUÇLARI")
    print("=" * 70)
    print(f"Toplam satır sayısı: {len(df_sample):,}")
    print(f"Benzersiz kategori sayısı: {df_sample['main_category'].nunique()}")
    print(f"Benzersiz ürün sayısı: {df_sample['parent_asin'].nunique():,}")
    print(f"Benzersiz kullanıcı sayısı: {df_sample['user_id'].nunique():,}")

    print(f"\nKategori başı yorum dağılımı:")
    cat_counts = df_sample['main_category'].value_counts().sort_index()
    for cat, count in cat_counts.items():
        print(f"  {cat[:40]:<40} {count:>10,}")

    print(f"\nRating dağılımı:")
    rating_counts = df_sample['rating'].value_counts().sort_index()
    for rating, count in rating_counts.items():
        print(f"  Rating_{rating}: {count:>10,}")


    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    full_output_path = os.path.join(output_dir, output_filename)

    print(f"\nCSV kaydediliyor: {full_output_path}")
    df_sample.to_csv(full_output_path, index=False, encoding='utf-8-sig')

    # Dosya boyutunu göster
    file_size_mb = os.path.getsize(full_output_path) / (1024 * 1024)
    print(f"Dosya boyutu: {file_size_mb:.1f} MB")

    print(f"\n{'='*70}")
    print("ÖRNEKLEM ÇEKME TAMAMLANDI")
    print(f"{'='*70}")

    return df_sample

In [ ]:
df_sample = create_main_category_sample(
    df_main_passed=df_main_passed,
    input_file_path=input_path,
    metadata_file_path=metadata_path,
    output_dir='/content/drive/MyDrive/Veri_madenciliği/Dataset/',
    output_filename='main_kategori_orneklem.csv',
    target_samples_per_rating=10000,
    min_reviews_per_product=5,
    max_reviews_per_product=50,
    excluded_ratings=[3],
    excluded_categories=['Apple Products'],
    random_seed=42
)

## GitHub için Örneklem Dosyası Oluşturma

Bu bölümde, bir önceki adımda oluşturulan ana kategori örneklem dosyasının ( `main_kategori_orneklem.csv`)  Githuba atmak için ilk 1000 satırını içeren yeni bir CSV dosyası oluşturuyoruz

In [ ]:
import pandas as pd
import os


original_sample_path = os.path.join(analysis_output_dir, 'main_kategori_orneklem.csv')


output_github_sample_filename = 'main_kategori_orneklem_github_1000_rows.csv'
output_github_sample_path = os.path.join(analysis_output_dir, output_github_sample_filename)

print(f"Orjinal örneklem dosyası yükleniyor: {original_sample_path}")

try:
    # Orjinal dosyayı oku
    df_full_sample = pd.read_csv(original_sample_path)


    df_github_sample = df_full_sample.head(1000)


    df_github_sample.to_csv(output_github_sample_path, index=False, encoding='utf-8-sig')

    print(f"\nİlk 1000 satırlık örneklem başarıyla oluşturuldu ve kaydedildi:\n{output_github_sample_path}")
    print(f"Satır sayısı: {len(df_github_sample):,}")

    print("\n--- GitHub Örneklemi (İlk 5 Satır) ---")
    display(df_github_sample.head())

except FileNotFoundError:
    print(f"Hata: Dosya bulunamadı - {original_sample_path}")
except Exception as e:
    print(f"Bir hata oluştu: {e}")